# One-Shard End-to-End Validation

This notebook validates the MVP against one full compressed brWaC shard. It prepares bronze segments, extracts silver comparison candidates, writes silver Parquet plus compact gold CSV outputs, displays aggregate charts, and records observed feasibility notes from the run.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "src" / "tal_qual").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/tal_qual")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

REPO_ROOT

In [ ]:
from pyspark.sql import SparkSession
import tempfile
import zipfile

spark = (
    SparkSession.builder
    .appName("tal-qual-one-shard-validation")
    .getOrCreate()
)

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))

spark.version

In [ ]:
from tal_qual import (
    BRONZE_OUTPUT_PATH,
    RAW_CORPUS_INPUT,
    SILVER_OUTPUT_PATH,
    connector_family_counts_dataframe,
    pattern_type_counts_dataframe,
    prepare_bronze_dataframe,
    prepare_silver_dataframe,
    sample_examples_dataframe,
    top_vehicles_global_dataframe,
    write_bronze_parquet,
    write_mvp_outputs,
)

input_path = REPO_ROOT / RAW_CORPUS_INPUT
bronze_output_path = REPO_ROOT / BRONZE_OUTPUT_PATH
silver_output_path = REPO_ROOT / SILVER_OUTPUT_PATH

assert input_path.exists(), f"Missing required one-shard input: {input_path}"

input_path, bronze_output_path, silver_output_path

## Build Bronze, Silver, And Gold Outputs

In [ ]:
bronze_df = prepare_bronze_dataframe(spark, input_path).cache()
bronze_count = bronze_df.count()
write_bronze_parquet(bronze_df, bronze_output_path)

bronze_count

In [ ]:
silver_df = prepare_silver_dataframe(bronze_df).cache()
candidate_count = silver_df.count()
write_mvp_outputs(silver_df, silver_output_path)

candidate_count

In [ ]:
from pyspark.sql.functions import col

connector_counts_df = connector_family_counts_dataframe(silver_df).cache()
pattern_counts_df = pattern_type_counts_dataframe(silver_df).cache()
top_vehicles_df = top_vehicles_global_dataframe(silver_df).cache()
examples_df = sample_examples_dataframe(silver_df).cache()

connector_counts_pdf = connector_counts_df.toPandas()
pattern_counts_pdf = pattern_counts_df.toPandas()
top_vehicles_pdf = top_vehicles_df.limit(20).toPandas()
examples_pdf = (
    examples_df
    .select(
        "connector_family",
        "pattern_type",
        "candidate_full_text",
        "vehicle_raw",
        "vehicle_normalized",
    )
    .limit(30)
    .toPandas()
)

connector_counts_pdf, pattern_counts_pdf.head(), top_vehicles_pdf.head()

## Visualizations

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
connector_counts_pdf.sort_values("count").plot.barh(
    x="connector_family",
    y="count",
    ax=ax,
    legend=False,
    color="#2a6f97",
)
ax.set_title("Candidates by connector family")
ax.set_xlabel("candidate count")
ax.set_ylabel("connector family")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
pattern_counts_pdf.sort_values("count").plot.barh(
    x="pattern_type",
    y="count",
    ax=ax,
    legend=False,
    color="#8f5d2a",
)
ax.set_title("Candidates by pattern type")
ax.set_xlabel("candidate count")
ax.set_ylabel("pattern type")
plt.tight_layout()

## Candidate Examples

In [ ]:
examples_pdf

## Observed Results And Limitations

In [ ]:
from IPython.display import Markdown, display

top_connectors = connector_counts_pdf.to_dict("records")
top_patterns = pattern_counts_pdf.to_dict("records")
top_vehicles = top_vehicles_pdf.head(10).to_dict("records")

display(Markdown(
    "\n".join([
        f"- Input shard: `{RAW_CORPUS_INPUT}`",
        f"- Bronze segment rows: `{bronze_count:,}`",
        f"- Silver candidate rows: `{candidate_count:,}`",
        f"- Connector-family counts: `{top_connectors}`",
        f"- Pattern-type counts: `{top_patterns}`",
        f"- Top normalized vehicles: `{top_vehicles}`",
        "- Known limitation: extraction is intentionally connector-based and does not classify literal versus figurative usage.",
        "- Known limitation: generic `como` remains excluded, so recall is deliberately lower in exchange for less noise.",
        "- Known limitation: `vehicle_raw` is a bounded right-context phrase, not a syntactic noun phrase or semantic vehicle.",
        "- Feasibility answer: the one-shard run produces enough connector and pattern signal for MVP inspection, so connector-based extraction is feasible on a meaningful brWaC subset.",
    ])
))